# Zero-Shot Topic Classification with BERTopic

Assigns each Korean K-pop article to one of four predefined categories using sentence-BERT embeddings,
with BERTopic's clustering pipeline disabled.

In [7]:
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from konlpy.tag import Okt
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.dimensionality import BaseDimensionalityReduction
from bertopic.cluster import BaseCluster

In [8]:
BASE_DIR = Path("../data")
INPUT_CSV = Path("../data/top300_filtered.csv")
OUTPUT_CSV = Path("../data/top300_filtered_with_topics_ZeroShot.csv")
ARTICLES_DIR = BASE_DIR

**Preprocessing.** Read each article and extract Korean nouns (≥2 chars) with Okt. Tokenized text gives
cleaner c-TF-IDF keywords than raw text.

In [9]:
def load_and_preprocess_documents(csv_path, articles_dir):
    """
    Loads metadata, reads article files, and tokenizes Korean
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found")

    df = pd.read_csv(csv_path)
    okt = Okt()
    processed_docs = []
    valid_indices = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        file_name = Path(row['file_path']).name
        full_path = articles_dir / file_name

        try:
            with open(full_path, 'r', encoding='utf-8') as f:
                text = f.read()
                
            # Tokenize using KoNLPy (Okt)
            tokens = okt.nouns(text)
            tokens = [word for word in tokens if len(word) > 1]
            processed_text = " ".join(tokens)
            
            processed_docs.append(processed_text)
            valid_indices.append(idx)
            
        except Exception as e:
            print(f"Could not read file {file_name}. Error: {e}")

    df_clean = df.loc[valid_indices].copy()
    return df_clean, processed_docs


In [10]:
def run_topic_modeling():
    df, documents = load_and_preprocess_documents(INPUT_CSV, ARTICLES_DIR)
    documents = [str(doc) for doc in documents]

    zeroshot_topic_list = [
        "스타", 
        "결혼 연애", 
        "사망", 
        "논란"
    ]

    # Manually set stop words
    korean_stop_words = ['통해', '단독', '♥', '라며', '사진', '기자', '방송', '뉴스', '텐아시아', '배우', '사람', '스포츠조선', '엑스포츠', 'OSEN', '소식', '스타뉴스', '마이데일리']
    
    vectorizer_model = CountVectorizer(stop_words=korean_stop_words)
    embedding_model = SentenceTransformer("jhgan/ko-sbert-sts")
    
    print("Encoding documents...")
    embeddings = embedding_model.encode(documents, show_progress_bar=True, batch_size=4)

    empty_dimensionality_model = BaseDimensionalityReduction()
    empty_cluster_model = BaseCluster()

    print("Fitting pure Zero-Shot BERTopic model...")
    topic_model = BERTopic(
        umap_model=empty_dimensionality_model,
        hdbscan_model=empty_cluster_model,
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model,
        zeroshot_topic_list=zeroshot_topic_list,   
        zeroshot_min_similarity=0.01, # Keep low to force assignment
    )
    
    topics, probs = topic_model.fit_transform(documents, embeddings=embeddings)

    df['topic'] = topics
    
    unique_topics = set(topics)
    num_topics = len(unique_topics) - 1 if -1 in unique_topics else len(unique_topics)
    print(f"Success: Found {num_topics} topics.")

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Results saved: {OUTPUT_CSV}")

    return topic_model, df

Topic IDs aren't guaranteed to match the order of `zeroshot_topic_list` — read the `Name`
column to see what each cluster actually is.

In [11]:
topic_model, final_df = run_topic_modeling()

display(topic_model.get_topic_info())

 50%|█████     | 151/300 [00:10<00:05, 28.35it/s]

Could not read file 20251027_article_1.txt. Error: [Errno 2] No such file or directory: '..\\data\\20251027_article_1.txt'


 94%|█████████▍| 282/300 [00:14<00:00, 29.69it/s]

Could not read file 20260205_article_2.txt. Error: [Errno 2] No such file or directory: '..\\data\\20260205_article_2.txt'


100%|██████████| 300/300 [00:15<00:00, 19.97it/s]


Encoding documents...


Batches: 100%|██████████| 75/75 [00:53<00:00,  1.41it/s]


Fitting pure Zero-Shot BERTopic model...
Success: Found 4 topics.
Results saved: ..\data\top300_filtered_with_topics_ZeroShot.csv


,Topic,Count,Name,Representation,Representative_Docs
0,0,150,0_박나래_매니저_공개_주장,"[박나래, 매니저, 공개, 주장, 논란, 모습, 영상, 사실, 자신, 대해]",[스타 뉴스 최혜진 기자 코미디언 박나래 사진 이동훈 코미디언 박나래 대한 자극 사...
1,1,64,1_공개_우리_모습_아시아,"[공개, 우리, 모습, 아시아, 멤버, 유튜브, 출연, 웃음, 조세호, 미니시리즈]",[스포츠조선 조윤선 기자 가수 서태지 크리스마스 근황 서태지 성탄절 이브 라며 장문...
2,2,56,2_결혼_결혼식_공개_부부,"[결혼, 결혼식, 공개, 부부, 아내, 이혼, 최준희, 지난, 모습, 유튜브]",[김종국 마이데일리 김진 기자 머리칼 하나 공개 지난해 결혼 김종국 꽁꽁 아내 정체...
3,3,28,3_사망_지난_선수_활동,"[사망, 지난, 선수, 활동, 김혜수, 향년, 세상, 원혁, 영화, 치료]",[우리 방금 결혼 스틸컷 스포츠조선 기자 할리우드 배우 브리트니 머피 사망 사망 경...
